In [1]:
from assetextractor.extraction.utils import Config

from assetextractor.parsing.core.assets import Asset, AssetCache
from assetextractor.parsing.core.templates import Template
from assetextractor.parsing.core.attributes import ListAttribute, DictAttribute, FileNameAttribute, ListItem

import json
from pathlib import Path
from lxml.etree import tostring

import typing as t
import re

In [2]:
include_dlcs_up_to = 2

In [3]:
config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates

In [4]:
class IconCache:
    def __init__(self):
        self.cache : dict[str, FileNameAttribute] = {}
        self.high_res : set[str] = set()
    
    def process(self, asset: Asset, is_high_res: bool = False) -> dict[str, t.Any]:
        result : dict[str, t.Any] = dict()
        result["guid"] = asset.guid
        result["name"] = asset.find_value("Standard.Name")
        icon = asset.find("Standard.IconFilename")
        if isinstance(icon, FileNameAttribute) and icon.is_image:
            self.cache[icon.identifier] = icon
            result["iconPath"] = icon.identifier
            
            if is_high_res:
                self.high_res.add(icon.identifier)
    
        if asset.text is not None:
            result["locaText"] = asset.text.values

        return result  

    def process_attribute(self, attr: DictAttribute) -> dict[str, t.Any]:
        result : dict[str, t.Any] = dict()
        result["id"] = attr.name
        result["name"] = attr.name

        icon = attr.find("Icon")
        if isinstance(icon, FileNameAttribute) and icon.is_image:
            self.cache[icon.identifier] = icon
            result["iconPath"] = icon.identifier
    
        if attr.Name() is not None:
            result["locaText"] = attr.Name().values

        return result  

    def to_dict(self):
        result : dict[str, str] = dict()
        for identifier, icon in self.cache.items():
            try:
                data_url = icon.get_data_url(25, 0.25 if identifier in self.high_res else 0.125)
                if data_url is None:
                    raise ValueError()

                result[identifier] = data_url
            except Exception as e:
                print(f"Conversion of {identifier} failed: {e}")

        return result

In [5]:
def flatten_pool(pool: Asset | None) -> list[int]:
    if pool is None:
        return []

    return sorted([asset.guid for asset in pool.pool_assets().keys()])

In [6]:
params: dict[str, t.Any] = dict()
params["constants"] = dict()
icons = IconCache()

schema: dict[str, t.Any] = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "Anno 117 Calculator Parameters",
    "description": "Generated schema for Anno 117 calculator parameters",
    "type": "object",
    "properties": {
        "constants": {
            "type": "object",
            "title": "Constants",
            "properties": {},
            "required": []
        }
    },
    "required": []
}

def infer_type_from_value(value: t.Any):
    """Infer JSON schema type from Python value."""
    if isinstance(value, bool):
        return "boolean"
    elif isinstance(value, int):
        return "integer"
    elif isinstance(value, float):
        return "number"
    elif isinstance(value, str):
        return "string"
    elif isinstance(value, list):
        return "array"
    elif isinstance(value, dict):
        return "object"
    else:
        return "string"

def generate_schema_for_value(value: t.Any, path: str=""):
    """Generate schema for a single value."""
    if isinstance(value, dict):
        properties = {}
        required = []
        
        for key, val in value.items():
            properties[key] = generate_schema_for_value(val, f"{path}.{key}")
            if val is not None:
                required.append(key)
        
        local_schema: dict[str, t.Any] = {
            "type": "object",
            "properties": properties
        }
        if required:
            local_schema["required"] = required
        return local_schema
        
    elif isinstance(value, list) and value:
        # Analyze all items in the array to determine schema
        if all(isinstance(item, dict) for item in value):
            # Collect all possible properties from all items
            all_properties = {}
            property_counts = {}
            
            for item in value:
                for prop_key, prop_val in item.items():
                    val_is_valid = prop_val is not None and (not isinstance(prop_val, list) or len(prop_val) > 0)
                    if prop_key not in all_properties and val_is_valid:
                        all_properties[prop_key] = generate_schema_for_value(prop_val, f"{path}[].{prop_key}")
                        property_counts[prop_key] = 0
                    if val_is_valid:
                        property_counts[prop_key] += 1
            
            # Determine required properties (present in all items)
            required_properties = [prop for prop, count in property_counts.items() if count == len(value)]
            
            item_schema: dict[str, t.Any] = {
                "type": "object",
                "properties": all_properties
            }
            if required_properties:
                item_schema["required"] = required_properties
        else:
            # For non-dict arrays, use the first item as before
            item_schema = generate_schema_for_value(value[0], f"{path}[0]")
        
        return {
            "type": "array",
            "items": item_schema
        }
        
    else:
        return {"type": infer_type_from_value(value)}

def add_constant(key: str, value: t.Any, description: str):
    params["constants"][key] = value
    schema["properties"]["constants"]["properties"][key] = {"type": infer_type_from_value(value), "description": description}
    schema["properties"]["constants"]["required"].append(key)

def add_parameter(key: str, value: list[t.Any] | dict[str,t.Any], class_name: str, description: str | None = None):
    """Add a parameter to the params dict with metadata."""    
    
    schema["properties"][key] = generate_schema_for_value(value, key)
    schema["properties"][key]["title"] = class_name
    if key not in schema["required"]:
        schema["required"].append(key)
                
    if  description:
        schema["properties"][key]["description"] = description
    
    params[key] = value

### DLCs and their unlock mechanic

In [7]:
# === DLCs ===
from assetextractor.parsing.core.attributes import ReferenceAttribute

uplay_template = templates["UplayProduct"]

# Build DLC prefix map: "dlc01" -> 67902, "dlc02" -> 67903, ...
# Source: UplayProduct.Standard.ID e.g. "DLC01_Prophecies_of_Ash"
dlc_prefix_map: dict[str, int] = {}

dlcs = []
if uplay_template:
    for dlc in list(uplay_template.assets):
        if(dlc.find_value("UplayProduct.ProductType") != "DLC"):
            continue
        
        is_allowed = True
        js = icons.process(dlc)
        try:
            dlc_id = dlc.find("Standard.ID")()
            if dlc_id:
                full_id = str(dlc_id)
                js["id"] = full_id
                prefix = full_id.split("_")[0].lower() if "_" in full_id else full_id.lower()
                
                # Filter by DLC number
                match = re.search(r"dlc(\d+)", prefix)
                if match:
                    dlc_num = int(match.group(1))
                    if dlc_num > include_dlcs_up_to:
                        is_allowed = False
                
                if is_allowed:
                    dlc_prefix_map[prefix] = dlc.guid
        except Exception:
            pass
        
        if is_allowed:
            dlcs.append(js)

add_parameter("dlcs", dlcs, "DLC", "DLC definitions with GUIDs and identifiers")

allowed_dlc_guids = set(dlc_prefix_map.values())

# Complete DLC map for detection: ALL DLCs regardless of include_dlcs_up_to.
# Detection must cover every DLC so above-threshold assets are correctly tagged
# and can be excluded from the output by the calculator.
all_dlc_prefix_map: dict[str, int] = {}
if uplay_template:
    for dlc in list(uplay_template.assets):
        if dlc.find_value("UplayProduct.ProductType") != "DLC":
            continue
        try:
            dlc_id = dlc.find("Standard.ID")()
            if dlc_id:
                full_id = str(dlc_id)
                prefix = full_id.split("_")[0].lower() if "_" in full_id else full_id.lower()
                all_dlc_prefix_map[prefix] = dlc.guid
        except Exception:
            pass

all_dlc_guids = set(all_dlc_prefix_map.values())

In [8]:
def get_unlocks(asset: Asset) -> list[int]:
    return sorted(asset.unlocked_by_dlcs.keys())


In [9]:
# Map DLC prefix -> region literal; extend here as new regional DLCs ship
_dlc_region_map = {
    "dlc01": "Roman",
    "dlc02": "Roman",
    "dlc03": "Egyptian",
}
dlc_region_filter = {
    all_dlc_prefix_map[p]: [r]
    for p, r in _dlc_region_map.items()
    if p in all_dlc_prefix_map
}

def filter_dlc_targets(js):
    # Filter targets for DLC effects
    if js["effectScope"].endswith("Session"):
        for dlc_guid, allowed_regions in dlc_region_filter.items():
            if dlc_guid in js["dlcUnlocks"]:
                filtered = []
                for t_guid in js["targets"]:
                    t_asset = assets.elements.get(t_guid)
                    if t_asset:
                        t_regions_attr = t_asset.find("Building.AssociatedRegions")
                        if t_regions_attr:
                            try:
                                t_regions = t_regions_attr()
                                if isinstance(t_regions, list):
                                    if any(r_id in allowed_regions for r_id in t_regions):
                                        filtered.append(t_guid)
                                elif t_regions in allowed_regions:
                                    filtered.append(t_guid)
                            except:
                                pass
                if len(filtered) == 0:
                    print(f"DLC region filtering would remove all targets for {js['guid']} ({js['name']})")
                js["targets"] = filtered
                break

### Constants

In [10]:
fuel = templates["EconomyFeature"].assets[0].EconomyFeature7.Fuel.Products[0]
add_constant("fuelProductionTime", fuel.ProductionTime().seconds, "How long does 1t of this product activate a production in seconds")
add_constant("fuelProduct", fuel.FuelProduct.guid, "which product can be used as fuel")

### Languages

In [11]:
add_parameter("languages", [l.lower() for l in assets.texts.languages.literals], "Language", "Supported localization languages")
#add_parameter("languages", ["english"], "Language", "Supported localization languages")

### Need Consumption

In [12]:
needConsumption = templates["DifficultyBalancing"].assets[0].DifficultySettings.NeedConsumption
factories = [{"id": cfg.name, "name": cfg.name, "locaText": cfg.ValueName().values, "consumptionFactor": cfg.ConsumptionFactor()} for cfg in needConsumption]
add_parameter("needConsumptions", factories, "NeedConsumption", "Scaling factor for NeedConsumptionRate")

### Regions

In [13]:
add_parameter("regions", [{**icons.process(asset), "id": asset.Region.RegionID(), "dlcUnlocks": get_unlocks(asset)} for asset in templates["Region"].assets], "Region", "Game regions data with GUID, name, icon, and localized text")

In [14]:
# Build fertility -> regions mapping from FertilitySet assets
fertility_regions = {}

for fset in templates["FertilitySet"].assets:
    region_attr = fset.find("ResourceSetCondition.AllowedRegion")
    if not region_attr:
        continue
    regions = region_attr()
    if not regions:
        continue
    fertilities_list = fset.find("FertilitySet.Fertilities")
    if not fertilities_list:
        continue
    for item in fertilities_list:
        ref = item.Fertility()
        if ref is None:
            continue
        if ref.template.name == "FertilityPool":
            pool_list = ref.find("FertilityPool.FertilityList")
            if pool_list:
                for pool_item in pool_list:
                    f = pool_item.Fertility()
                    if f:
                        fertility_regions.setdefault(f.guid, set()).update(regions)
        else:
            fertility_regions.setdefault(ref.guid, set()).update(regions)

fertilities_data = [{**icons.process(asset), "dlcUnlocks": get_unlocks(asset), "regions": sorted(fertility_regions.get(asset.guid))} for asset in templates["Fertility"].assets if fertility_regions.get(asset.guid) != None]
add_parameter("fertilities", fertilities_data, "Fertility", "Island fertililities and deposits that farms/mines require to operate. Referenced by factories (neededFertility) and building buffs (addedFertility).")

### Sessions (Require manual adjustment to include new ones)

In [15]:
#add_parameter("sessions", [icons.process(assets[guid]) for guid in [3225, 6627]], "Session", "Game session information including region associations")
result: list[t.Any] = []
regions_without_sessions: set[int] = set()
for region in params["regions"]:
    regions_without_sessions.add(region["guid"])

for guid in [37135, 3245, 6627, 149679]:
    asset = assets[guid]

    if asset is None:
         continue

    session = icons.process(asset) 
    session["dlcUnlocks"] = get_unlocks(asset)
    session["region"] = asset.find("Session.Region").guid
    regions_without_sessions.remove(session["region"])
    result.append(session)

add_parameter("sessions", result, "Session", "Game session information including region associations")

if len(regions_without_sessions) != 0:
    for guid in regions_without_sessions:
        region = next(r for r in params["regions"] if r["guid"] == guid)
        print(f"Region without session: GUID {guid}, Name: {region['name']}")

### Attributes

In [16]:
dict_attr: DictAttribute = templates["NeedAttributeFeature"].assets[0].NeedAttributeFeature.NeedAttributeConfig
result = [icons.process_attribute(attr) for attr in dict_attr]
add_parameter("needAttributes", result, "NeedAttribute", "Attributes obtained from needs")

### Needs

In [17]:
dict_attr: DictAttribute = templates["NeedCategoryConfig"].assets[0].NeedCategoryConfig.NeedCategorys
result = [icons.process_attribute(attr) for attr in dict_attr]
add_parameter("needCategories", result, "NeedCategory", "Categories for needs")

In [18]:
result = []
for asset in templates["Need"].assets:
    js = icons.process(asset)
    cfg = asset.Need
    js["needProduct"] = cfg.NeedProduct.guid
    js["needCategory"] = cfg.NeedCategoryType()
    js["supplyWeight"] = cfg.SupplyWeight()
    js["isBuilding"] = cfg.NeedProduct().Product.IsAbstract()
    js["needAttributes"] = dict()
    for attr in cfg.NeedAttributes:
        js["needAttributes"][attr.name] = int(attr.Value())
    js["dlcUnlocks"] = get_unlocks(asset)
    result.append(js)
add_parameter("needs", result, "Need", "Population group definitions with associated levels")

### Population Groups, Levels, and Residences

In [19]:

result = []
for asset in templates["PopulationGroup7"].assets:
    js = icons.process(asset)
    
    js["dlcUnlocks"] = get_unlocks(asset)
    js["populationLevels"] = [level.Level.guid for level in asset.find("PopulationGroup7.PopulationLevels")]
    js["region"] = asset.find("PopulationGroup7.Regional")()

    result.append(js)
add_parameter("populationGroups", result, "PopulationGroup", "Population group definitions with associated levels")

In [20]:
result = []
level_to_region : dict[int, str]= dict()
for asset in templates["ResidenceBuilding"].assets:
    js = icons.process(asset)
    
    js["dlcUnlocks"] = get_unlocks(asset)
    js["associatedRegions"] = asset.Building.AssociatedRegions()
    js["possibleUpgrades"] = [upgrade.UpgradeGUID.guid for upgrade in asset.Upgradable.PossibleUpgrades]
    
    cfg = asset.Residence7
    js["populationLevel"] = cfg.PopulationLevel.guid

    level_to_region[js["populationLevel"]] = js["associatedRegions"]

    js["needsList"] = []
    for need in cfg.NeedsList:
        js["needsList"].append({
            "need": need.Need.guid,
            "needConsumptionRate": None if need.NeedConsumptionRate() == 0 else need.NeedConsumptionRate()
        })

    result.append(js)
add_parameter("residenceBuildings", result, "ResidenceBuilding", "Residence building for a population level")

### Population Levels

In [21]:
result = []
for asset in templates["PopulationLevel"].assets:
    js = icons.process(asset, is_high_res=True)
    
    cfg = asset.PopulationLevel
    js["dlcUnlocks"] = get_unlocks(asset)
    js["connectedWorkforce"] = cfg.ConnectedWorkforce.guid
    js["populationToWorkforceFactor"] = cfg.PopulationToWorkforceFactor()
    js["associatedRegions"] = level_to_region[js["guid"]]
    result.append(js)
add_parameter("populationLevels", result, "PopulationLevel", "Population level definitions with needs and requirements")

### Products and Workforce

In [22]:

products = []
workforce = []
for asset in templates["Product"].assets:
    if asset.Product.IsWorkforce():
        workforce.append({**icons.process(asset, is_high_res=True), "associatedRegions": asset.Product.AssociatedRegion(), "dlcUnlocks": get_unlocks(asset)})
    else:
        category = asset.Product.ProductCategory()
        products.append({**icons.process(asset, is_high_res=True), 
        "associatedRegions": asset.Product.AssociatedRegion(), 
        "isAbstract": asset.Product.IsAbstract(),
        "isConstructionMaterial": category is not None and category() == "Construction Material" and asset.guid != 2180,
        "dlcUnlocks": get_unlocks(asset)})

add_parameter("products", products, "Product", "Product definitions with producer information")
add_parameter("workforce", workforce, "Workforce", "Product definitions with producer information")

### Product Filter Meta

In [23]:
result = []

# First, collect all products from category index 0 (the "all products" category)
all_products_category = assets[28749].ProductFilter.Categories[0]
all_products = set(p.Product.guid for p in all_products_category.ProductList().ProductList.List)

# Collect products already assigned to categories
assigned_products = set()
for c in assets[28749].ProductFilter.Categories:
    if c.index == 0:
        continue

    category = {}
    category["iconPath"] = icons.process(c.Icon())["iconPath"]
    category["locaText"] = c.Text().values
    category["guid"] = c.ProductList.guid
    category["products"] = [p.Product.guid for p in c.ProductList().ProductList.List]
    
    # Track which products are already assigned
    assigned_products.update(category["products"])
    
    result.append(category)

# Find missing products (in all_products but not in any category)
missing_products = all_products - assigned_products

# Check other ProductList templates for additional products
for asset in templates["ProductList"].assets:
    if asset.guid != 28749:  # Skip the main ProductFilter asset
        other_products = set(p.Product.guid for p in asset.ProductList.List)
        # Add products that aren't already assigned to any category
        missing_products.update(other_products - assigned_products)

# Print missing products with their details
print("Missing products:")
for guid in sorted(missing_products):
    product_asset = assets[guid]
    if product_asset is not None:
        name = product_asset.find("Standard.Name")()
        english_text = product_asset.text.values.get("english", "") if product_asset.text else ""
        print(f"  GUID {guid}: {name} - {english_text}")

add_parameter("productFilters", result, "ProductFilter", "Product category filters for UI organization")

def add_missing(guid: int, category_index: int):
    if not guid in missing_products:
        return
    
    product_asset = assets[guid]
    if product_asset is not None:
        name = product_asset.find("Standard.Name")()
        english_text = product_asset.text.values.get("english", "") if product_asset.text else ""
        result[category_index]['products'].append(guid)
        print(f"Appended {guid}: {name} - {english_text} to {result[category_index]['locaText']['english']}")


Missing products:


### Factories

In [25]:
factories = []
modules = []
producers: dict[int, list[int]] = {}

# Track products used in factories
factory_input_products = set()
factory_output_products = set()

def process_list(lis: ListAttribute):
    res: list[dict[str, int]] = []

    for item in lis:
        res.append({
            "product": item.Product.guid,
            "amount": item.Amount()
        })
    
    return res

for template in templates.groups["Objects"]["Buildings"]["Factories"]:
    if not isinstance(template, Template) or template.name == "Monument":
        continue
    
    for asset in template.assets:

        if "TEST" in asset.name:
            continue

        factory = icons.process(asset)
        factory["associatedRegions"] = asset.find("Building.AssociatedRegions")()

        inputs = asset.find("FactoryBase.FactoryInputs")
        factory["inputs"] = process_list(inputs)

        # Track input products
        for input_item in factory["inputs"]:
            factory_input_products.add(input_item["product"])

        factory["needsFuelInput"] = asset.find("FactoryBase.NeedsFuelInput")()

        outputs = asset.find("FactoryBase.FactoryOutputs")
        factory["outputs"] = process_list(outputs)

        # Track output products
        for output_item in factory["outputs"]:
            factory_output_products.add(output_item["product"])

        for o in factory["outputs"]:
            guid = o["product"]
            if guid in producers:
                producers[guid].append(asset.guid)
            else:
                producers[guid] = [asset.guid]

        factory["maintenances"] = process_list(asset.Maintenance.Maintenances)

        factory["cycleTime"] = asset.find("FactoryBase.CycleTime")()

        fertility_attr = asset.find("Factory7.NeededFertility")
        factory["neededFertility"] = fertility_attr.guid if fertility_attr is not None and fertility_attr.guid else 0
        factory["dlcUnlocks"] = get_unlocks(asset)

        modules_limit = asset.find("ModuleOwner.ModuleLimits.Main.Limit")
        factory["modulesLimit"] = 0 if modules_limit is None else modules_limit()

        attr_aqueduct = asset.find("AqueductConsumer.AqueductConsumptionBuffEffect")
        if attr_aqueduct is not None and len(attr_aqueduct._value_list) > 0:
            factory["aqueductProductivityBuff"] = attr_aqueduct[0].ProductivityBuff.guid

        effect_attr = asset.find("Building.FunctionalEffects")
        if effect_attr is not None and len(effect_attr._value_list) > 0:
            factory["buffs"] = [buff.GUID.guid for item in effect_attr for buff in item.FunctionalEffect().Effect.Buffs ]

        module_attr = asset.find("ModuleOwner.AdditionalModule")
        if module_attr is not None:
            factory["additionalModule"] = module_attr.guid

        if template.name == "ProductionModuleSilo":
            modules.append(factory);
        else:
            if len(factory["outputs"]) == 0:
                raise ValueError(f"Factory {str(asset)} with GUID {asset.guid} of template {template.name} produces no goods.")
            
            factories.append(factory)

add_parameter("factories", factories, "Factory", "Factory building definitions with inputs, outputs, maintenances, and production rates")
add_parameter("modules", modules, "Module", "Modules attachable to factories with inputs, maintenances, and production rates")

for product in params["products"]:
    if product["guid"] in producers:
        factories = producers[product["guid"]]
        product["producers"] = factories

# Check which factory products are not in product filters
all_factory_products = factory_input_products | factory_output_products
filter_products = set()
for category in params["productFilters"]:
    filter_products.update(category["products"])

missing_from_filters = all_factory_products - filter_products

if len(missing_from_filters):
    print("Factory products not in product filters:")
    for guid in sorted(missing_from_filters):
        product_asset = assets[guid]
        if product_asset is not None:
            name = product_asset.find("Standard.Name")()
            english_text = product_asset.text.values.get("english", "") if product_asset.text else ""
            usage = []
            if guid in factory_input_products:
                usage.append("input")
            if guid in factory_output_products:
                usage.append("output")
            print(f"  GUID {guid}: {name} - {english_text} (used as: {', '.join(usage)})")

### Building Buffs

In [26]:
# print potentially new buffs or properties
diff = set(prop.name for prop in templates.groups["EffectSystem"]["Buffs"] if len(prop.assets)) - set(['TroopBuff', 'MetaBuff', 'DefenseBuildingBuff', 'AreaBuff', 'ShipBuff', 'BuildingBuff'])
if len(diff):
    print("New buff type", diff)

diff = set(prop.name for prop in templates["BuildingBuff"]) - set(['BuildingUpgrade', 'CityInstitutionUpgrade', 'IncidentInfectableUpgrade', 'Text', 'ModuleOwnerUpgrade', 'Standard', 'MaintenanceUpgrade', 'Buff', 'DistributionUpgrade', 'HealthUpgrade', 'IrrigationUpgrade', 'AqueductUpgrade', 'ResidenceUpgrade', 'WarehouseUpgrade', 'RecruitmentUpgrade', 'FactoryUpgrade'])
if len(diff):
    print("New BuildingBuff property", diff)

diff = set(prop.name for prop in assets.properties["BuildingUpgrade"]) - set(['AdditionalFunctionalEffect', 'AdditionalAttributes', 'AttributeModifierInPercent', 'AdditionalWorkforces', 'WorkforceModifierInPercent'])
if len(diff):
    print("New BuildingUpgrade attribute", diff)

diff = set(prop.name for prop in assets.properties["ResidenceUpgrade"]) - set(['ConsumptionModifierInPercent',
 'GoodConsumptionUpgrade',
 'NeedProvidedNeedAttributes',
 'ProvidedNeedUpgrade'])
if len(diff):
    print("New ResidenceUpgrade attribute", diff)

diff = set(prop.name for prop in assets.properties["FactoryUpgrade"]) - set(['AddedFertility','AddedAreaFertility', 'AreaFertilityPercent', 'InputAmountUpgrade', 'CanUseMarsh', 'NeededAreaUpgrade', 'AdditionalOutput', 'FertilityPercent', 'InfluenceRadiusUpgrade', 'CanUseForest', 'ReplaceInputs', 'ProductivityUpgrade', 'FuelDurationPercent', 'CanUseMeadow'])
if len(diff):
    print("New FactoryUpgrade attribute", diff)

diff = set(prop.name for prop in assets.properties["ModuleOwnerUpgrade"]) - set(['ModuleLimitPercent'])
if len(diff):
    print("New ModuleOwnerUpgrade attribute", diff)

diff = set(prop.name for prop in assets.properties["MaintenanceUpgrade"]) - set(['ReplaceWorkforce', 'MaintenanceFactorUpgrade', 'WorkforceMaintenanceFactorUpgrade', 'EncampedUnitScalingFactorUpgrade'])
if len(diff):
    print("New MaintenanceUpgrade attribute", diff)

New buff type {'WarehouseBuff', 'ForwardBuff'}
New BuildingBuff property {'RaceTrackUpgrades'}
New ResidenceUpgrade attribute {'AdditionalNeedsDemand'}


In [27]:
for prop in ["BuildingUpgrade","ResidenceUpgrade","FactoryUpgrade", "MaintenanceUpgrade"]:
    assets.properties[prop].print_tree()

BuildingUpgrade:
	AdditionalFunctionalEffect: Asset
	AdditionalAttributes: 
		AmountOrPercent: FloatOrPercental
	AttributeModifierInPercent: Float
	WorkforceModifierInPercent: Float
	AdditionalWorkforces: 
		WorkforceGUID: Asset
*) inherited **) default
ResidenceUpgrade:
	ProvidedNeedUpgrade: 
		ProvidedNeed: Asset
	GoodConsumptionUpgrade: 
		ProvidedNeedProduct: Asset
		AmountInPercent: Float
	ConsumptionModifierInPercent: Float
	NeedProvidedNeedAttributes: 
		ChangeNeedAttributesOf: 
			ProvidedProduct: Asset
		AdditionalNeedAttributes: 
			AmountOrPercent: FloatOrPercental
	AdditionalNeedsDemand: 
		Needs: 
			Need: Asset
*) inherited **) default
FactoryUpgrade:
	ProductivityUpgrade: Upgrade
	AddedFertility: Asset
	FertilityPercent: Integer
	AdditionalOutput: 
		Product: Asset
		ForceProductSameAsFactoryOutput: Boolean
		AdditionalOutputCycle: Integer
		Amount: Integer
		RequiresLimitedLodeFertility: Asset
		EffectScaleApplyToCycles: Boolean
		EffectScaleApplyToAmount: Boolean
	Inpu

In [28]:
from assetextractor.parsing.core.attributes import PrimitiveAttribute,UpgradeAttribute
ignored_buffs = [95767, # Cheat Boost
                51013, # Maria Bacca (no loca or icon)
                147993, 147997, 147999, 148001, 148003, # Stockpiling for Disaster
                148378, 148380, 148383, 148385, 148389, 148391, # Restoring the Pax Deorum
                147485, 147487, 147492, 147495, # Edict of Public Service
                147942, 147940, 147948, 150370, 147950, 150372, # After the Storm
                148169, 150411, 148162, 148166, 148167, 150419, 148237, # Panic in the Streets
                149299, 150481, 150483, 150487, 150485, # A Harvest of Dark Glass
                106570, 106572, # Ocus Iad buffs from quest events
                140764, 98692, # Space for the displaced buff from quest events
                107479, 140884, # Bread Sentries
                156981, # swortd to ploughshares
                148427, 148428, # Caeso's Slaves
                145370, # DLC01 Events
                153838, 153839, 153840, 153841, 153842, 153843, 153844, # Hippodrome levels 02-09
                155521, 155523, # DLC02 events
                166422, # DLC03 Specialist with invalid additional outputs target
                ] 

# hardcoded buffs
base_productivity_upgrade = {
    43610: 50
}

stackable_buffs = [
    82018, # deep mines
    145100, # fertile soil
    145096, 148044 # obsidian gathering
]

result = []
relevant_buffs : set[int] = set()
population_buffs : set[int] = set()



# upgrade properties
upgrade_paths = {
    "BuildingUpgrade.AdditionalWorkforces": "additionalWorkforces",
    "BuildingUpgrade.WorkforceModifierInPercent": "workforceModifierInPercent",
    "FactoryUpgrade.AdditionalOutput": "additionalOutputs",
    "FactoryUpgrade.ProductivityUpgrade": "productivityUpgrade",
    "FactoryUpgrade.FuelDurationPercent": "fuelDurationPercent",
    "FactoryUpgrade.ReplaceInputs": "replaceInputs",
    "FactoryUpgrade.AddedFertility": "addedFertility",
    "FactoryUpgrade.FertilityPercent": "fertilityPercent",
    "MaintenanceUpgrade.ReplaceWorkforce": "replaceWorkforce",
    "MaintenanceUpgrade.WorkforceMaintenanceFactorUpgrade": "workforceMaintenanceFactorUpgrade",
    "BuildingUpgrade.AdditionalAttributes.Population.AmountOrPercent": "population",
    "ResidenceUpgrade.AdditionalNeedsDemand.Needs": "additionalNeedsDemand",
    "ResidenceUpgrade.ProvidedNeedUpgrade": "providedNeedUpgrade",
    "ResidenceUpgrade.ConsumptionModifierInPercent": "consumptionModifierInPercent",
    "ResidenceUpgrade.GoodConsumptionUpgrade": "goodConsumptionUpgrade",
}

unhandeled_paths = [
    "FactoryUpgrade.InputAmountUpgrade",
    "FactoryUpgrade.MaxWorkerAmountUpgrade",
]

for asset in templates["BuildingBuff"].assets:
    if asset.guid in ignored_buffs:
        continue

    if "DEPRECATED" in asset.name or "UNUSED" in asset.name:
        continue

    icon = asset.find("Standard.IconFilename")
    if not isinstance(icon, FileNameAttribute) or not icon.is_image:
        continue
    
    # Check for relevant properties
    has_relevant_properties = False  

    if asset.guid in base_productivity_upgrade:
        has_relevant_properties = True

    # Check each path for meaningful values
    for path, key in upgrade_paths.items():
        try:
            attribute = asset.find(path)
            if attribute is not None:
                if isinstance(attribute, ListAttribute) and len(attribute._value_list) > 0:
                    has_relevant_properties = True
                    break
                elif (isinstance(attribute, PrimitiveAttribute) or isinstance(attribute, UpgradeAttribute)) and attribute() != 0.0:
                    if key == "population":
                        if attribute() > 0 and "Mythic" not in asset.find_value("Standard.Name"): # Exclude population buffs of mythic items, since we need to consider (i) boosting, (ii) radius, (iii) additional effects like supplied need (frequent) and reduced bread consumption (item 160084 - Pantites of Achaea)
                            has_relevant_properties = True
                            population_buffs.add(asset.guid)
                        break
                    
                    if not (key == "productivityUpgrade" and attribute() < 0 or # Exclude negative productivity upgrades from catastrophe events
                            key == "workforceModifierInPercent" and attribute() > 0 or # Exclude more workforce required from catastrophe events
                            key == "consumptionModifierInPercent" and attribute() > 0 or # Exclude more consumption required from catastrophe events
                            key == "fertilityPercent"):  #AddedFertility is only relevant
                        has_relevant_properties = True
                        break
                    
                elif "OldWorkforce" in attribute and attribute.OldWorkforce(): # ReplaceWorkforce
                    has_relevant_properties = True
                    break
                elif key == "addedFertility" and hasattr(attribute, "guid") and attribute.guid: # AddedFertility
                    has_relevant_properties = True
                    break
        except Exception as e:
            print(asset, asset.guid, e)
            continue

    if not str(asset).startswith("Nearby"): # Exclude effects that exist to fullfill needs from public services
        for path in unhandeled_paths:
            attribute = asset.find(path)
            if attribute is not None:
                if isinstance(attribute, ListAttribute) and len(attribute._value_list) > 0:
                    print(asset, asset.guid, path, attribute(), "not handeled")
                elif (isinstance(attribute, PrimitiveAttribute) or isinstance(attribute, UpgradeAttribute)) and attribute() != 0.0:
                    has_relevant_properties = True
                    print(asset, asset.guid, path, attribute(), "not handeled")

    if has_relevant_properties:
        buff = icons.process(asset) 
        buff["isStackable"] = asset.Buff.IsStackable() or buff["guid"] in stackable_buffs

        buff["baseProductivityUpgrade"] = base_productivity_upgrade[asset.guid] if asset.guid in base_productivity_upgrade else 0

        # Add building upgrade properties
        for path, key in upgrade_paths.items():
            try:
                attribute = asset.find(path)

                if key == "additionalWorkforces":
                    buff[key] = [workforce.WorkforceGUID.guid for workforce in attribute]
                elif key == "additionalOutputs":
                    buff[key] = [{
                                "product": extra.Product.guid,
                                "forceProductSameAsFactoryOutput": extra.ForceProductSameAsFactoryOutput(),
                                "additionalOutputCycle": extra.AdditionalOutputCycle(), 
                                "amount": extra.Amount() 
                    } for extra in attribute]
                elif key == "replaceInputs":
                    buff[key] = [{
                                "newInput": entry.NewInput.guid,
                                "oldInput": entry.OldInput.guid,
                    } for entry in attribute]
                elif key == "addedFertility":
                    buff[key] = attribute.guid if attribute is not None else 0
                elif key == "fertilityPercent":
                    buff[key] = int(attribute()) if attribute is not None and buff.get("addedFertility", 0) != 0 else 100
                elif key == "replaceWorkforce":
                    buff[key] = {
                                "newWorkforce": attribute.NewWorkforce.guid,
                                "oldWorkforce": attribute.OldWorkforce.guid,
                    }
                elif key == "additionalNeedsDemand":
                    buff[key] = [entry.Need.guid for entry in attribute]
                elif key == "providedNeedUpgrade":
                    buff[key] = [entry.ProvidedNeed.guid for entry in attribute]
                elif key == "goodConsumptionUpgrade":
                    buff[key] = [{
                                "product": entry.ProvidedNeedProduct.guid,
                                "amountInPercent": entry.AmountInPercent(),
                    } for entry in attribute]
                else:
                    buff[key] = attribute()
            except:
                continue
  
        buff["dlcUnlocks"] = get_unlocks(asset)
        relevant_buffs.add(asset.guid)
        result.append(buff)

add_parameter("buildingBuffs", result, "BuildingBuff", "Building buff assets with upgrade properties for buildings, residences, and factories. Determine what the effect is. See effects for where they apply.")

schema["properties"]["buildingBuffs"]["items"]["properties"]["replaceInputs"] = generate_schema_for_value([{
    "newInput": 1,
    "oldInput": 1
}])

In [29]:
def filter_effect_targets(js):
    # Filter targets to only residences for Population-only effects
    residence_guids = set(r["guid"] for r in params.get("residenceBuildings", []))
    if any(b_guid in population_buffs for b_guid in js["buffs"]):
        js["targets"] = [t_guid for t_guid in js["targets"] if t_guid in residence_guids]

    # Filter targets by region of the source building
    asset = assets.elements.get(js["guid"])
    if not asset: return
    
    source_regions = set()
    for ref in asset.referenced_by.values():
        source_asset = ref.source
        regs = source_asset.find_value("Building.AssociatedRegions")
        if regs:
            if isinstance(regs, list):
                source_regions.update(regs)
            elif regs:
                source_regions.add(regs)

            
    if source_regions:
        filtered = []
        for t_guid in js["targets"]:
            t_asset = assets.elements.get(t_guid)
            if not t_asset:
                filtered.append(t_guid)
                continue
            
            t_regs = t_asset.find_value("Building.AssociatedRegions")
            if t_regs:                
                if isinstance(t_regs, list):
                    if any(r in source_regions for r in t_regs):
                        filtered.append(t_guid)
                elif t_regs in source_regions:
                    filtered.append(t_guid)
            else:
                filtered.append(t_guid)
        js["targets"] = filtered


### Area Buffs

In [30]:
ignored_area_buffs = [
    38420, # Latin Weave (Flax)
    38414, # Natural Resinance (Sandarac)
    38413, # Olive-Grower Wisdom (Olives)
    38422, # Wild Barley (Barley)
    38421, # Celtic Herbalism (Herbs)
    38426, # Saltwort Wisdom (Samphire)
]

result = []

for asset in templates["AreaBuff"].assets:
    if asset.guid in ignored_area_buffs:
        continue

    icon = asset.find("Standard.IconFilename")
    if not isinstance(icon, FileNameAttribute) or not icon.is_image:
        continue

    added_fertility = asset.find("AreaBuff.AddedAreaFertility")
    if added_fertility is None or not hasattr(added_fertility, "guid") or not added_fertility.guid:
        continue

    buff = icons.process(asset)

    buff["addedFertility"] = added_fertility.guid
    fertility_percent = asset.find("AreaBuff.AreaFertilityPercent")
    buff["fertilityPercent"] = int(fertility_percent()) if fertility_percent is not None else 100

    buff["dlcUnlocks"] = get_unlocks(asset)
    relevant_buffs.add(asset.guid)
    result.append(buff)

add_parameter("areaBuffs", result, "AreaBuff", "Area buff assets that add a fertility to all buildings in range.")

### Effects

In [31]:
def process_effect_scope(scope: str):
    if scope.endswith("Session"): return "session-event"  
    if scope == "ModuleOwner": return "module" 
    if scope == "StreetDistance" or scope == "Radius": return "building"
    return "island-event"

In [32]:
ignored_effects = [
    80865, # Epona population buff on production buildings
    82990, # workfoce upgrade for warehouses
    107478, 107622, # Workforce increase (bad outcome from event)
    82164, 97812, # clever carpentry and hand and heart (removed techs)
    118552, # unused Mercury Festival Gold Production
    97795, # Effect Productivity Hand and Heart deperecated 
    49804, # Effect Prydein Legendary Mines (removed)
    147992, 147996, 147998, 148000, 148002, # Stockpiling for Disaster
    148377, 148379, 148382, 148384, 148388, 148390, # Restoring the Pax Deorum
    147489, 147494, # Edict of Public Service
    147941, 147939, 147947, 150369, 147949, 150371, # After the Storm
    148168, 150402, 148161, 148165, 150418, 148236, # Panic in the Streets
    149298, 150478, 150482, 150486, 150484, # A Harvest of Dark Glass
    148045, 148078, 148049, 148053, 148068, 148072, # The Literal Aftermath
    148427, # Caeso's Slaves
    107937, 51354, # Amphitheatre Fame (outdated)
    68757, 68758, # Amphitheatre Splendour I, II (base game)
    153837, 157661, 157663, 157665, # Amphitheatre Splendour II, III, IV (no added need)
    157667, 157669, 157671, 157673, 157675, # Amphitheatre Splendour V-IX
    153882, # Hippodrome Splendour II
]

relevant_effects : set[int] = set()
result= []
effect_json : dict[int,any] = dict()

regex_all_buildings = r".*All (Production )?Buildings.*"

# Boost buffs of an ItemWithBoost live on the *item*: Effect.Buffs are the base
# (radius) buffs, ItemWithBoost.BoostBuffs the boosted ones. When a base/boost buff
# delegates its real upgrade to a nested BuildingUpgrade.AdditionalFunctionalEffect,
# the base and boost variants surface here as two *separate* Effect assets (e.g.
# "... Functional Effect" and "... Functional Effect Boosted") -- the source of the
# mythical "effect appears twice" duplication. Pair them through the item (base and
# boost buffs are index-aligned) so the effect is emitted once with an optional
# boostBuffs list and the boosted twin is dropped.
AFE_PATH = "BuildingUpgrade.AdditionalFunctionalEffect"
boost_effect_buffs: dict[int, list[int]] = {}   # base functional effect guid -> boost buff guids
boosted_functional_effects: set[int] = set()     # boost functional effect guids to drop

def _effect_buff_refs(list_attr):
    refs = []
    if isinstance(list_attr, ListAttribute):
        for entry in list_attr:
            ref = entry.find_ref("GUID")
            if ref is not None:
                refs.append(ref)
    return refs

for boost_item in templates["ItemWithBoost"].assets:
    base_buffs = _effect_buff_refs(boost_item.find("Effect.Buffs"))
    boost_buffs_list = _effect_buff_refs(boost_item.find("ItemWithBoost.BoostBuffs"))
    for i, boost_buff in enumerate(boost_buffs_list):
        if i >= len(base_buffs):
            break
        base_eff = base_buffs[i].find_ref(AFE_PATH)
        boost_eff = boost_buff.find_ref(AFE_PATH)
        if base_eff is None or boost_eff is None:
            continue
        boosted_functional_effects.add(boost_eff.guid)
        boost_buff_guids = []
        for boost_entry in boost_eff.Effect.Buffs:
            boost_ref = boost_entry.find_ref("GUID")
            if boost_ref is not None and boost_ref.guid in relevant_buffs:
                boost_buff_guids.append(boost_ref.guid)
        if boost_buff_guids:
            boost_effect_buffs[base_eff.guid] = boost_buff_guids

for asset in templates["Effect"].assets:
    if asset.guid in ignored_effects:
        continue

    if asset.guid in boosted_functional_effects:
        continue  # merged into its base variant's boostBuffs (single-row tri-state)

    try:
        cfg = asset.Effect
        buffs = [buff.GUID.guid for buff in cfg.Buffs if buff.GUID.guid  in relevant_buffs]
        
        if cfg.EffectScope() == "ModuleOwner":
            continue
        
        if len(buffs):

            js = icons.process(asset)
            js["buffs"] = buffs
            boost_buffs = boost_effect_buffs.get(asset.guid)
            if boost_buffs:
                js["boostBuffs"] = boost_buffs
            js["targets"] = list(dict.fromkeys(guid for pool in cfg.Targets for guid in flatten_pool(pool.find_ref("GUID"))))
            if (len(js["targets"]) == 0):
                continue 

            js["targetsIsAllProduction"] = any([re.match(regex_all_buildings, pool.GUID().Standard.Name()) for pool in cfg.Targets if pool.GUID()])
            if js["targetsIsAllProduction"]:
                print(js["guid"], js["name"])
            
            js["excludeEffectSourceGUID"] = cfg.ExcludeEffectSourceGUID()
            js["effectDuration"] = cfg.TimedEffect.EffectDuration().seconds

            if len(cfg.ExcludeFromTargets._value_list) > 0:
                print(asset.name, "ExcludeFromTargets", cfg.ExcludeFromTargets())

            js["effectScope"] = cfg.EffectScope()
            js["source"] = process_effect_scope(js["effectScope"])


            js["dlcUnlocks"] = get_unlocks(asset)

            filter_dlc_targets(js)
            filter_effect_targets(js)

            relevant_effects.add(asset.guid)
            effect_json[asset.guid] = js
            result.append(js)
    except Exception as e:
        print(asset.guid, e)

add_parameter("effects", result, "Effect", "Building buff combined with targets (pool of building GUIDs). Source is an enum with literals 'module', 'tech', 'festival', 'veneration-effect', 'session-event', 'island-event', 'building'")

174824 Effect Hippodrome Side Reward Tier 09 FunctionalEffect


In [33]:
# Villa-allocation Mythic items with a production-relevant island-wide MythicEffect.
# Derived dynamically instead of a hardcoded GUID list, so new Mythic Villa items
# added by future DLCs/patches are picked up automatically: Item.Rarity=="Mythic",
# Item.Allocation=="Villa", a resolvable MythicEffect whose Effect.EffectScope==
# "ObjectsInArea", and at least one Buff that made it into relevant_buffs (i.e. has
# a production-relevant field on its resolved BuildingBuff, per the upgrade_paths
# handled above).
mythical_item_effects = []
for asset in templates["ItemWithBoost"].assets:
    try:
        if asset.Item.Rarity() != "Mythic" or asset.Item.Allocation() != "Villa":
            continue

        mythic_effect = asset.find_ref("Item.MythicEffect")
        if mythic_effect is None:
            continue

        effect_json[mythic_effect.guid]["source"] = "mythical-item"

        mythical_item_effects.append(effect_json[mythic_effect.guid])
    except Exception as e:
        print(asset.guid, e)

print(f"Found {len(mythical_item_effects)} production-relevant Mythic Villa item effects")

160063 166449
160066 166451
160072 166453
160078 166457
160084 166461
160087 166463
160090 166465
160093 166467
160491 166471
160494 166473
160522 166489
160525 166491
160528 166493
Found 17 production-relevant Mythic Villa item effects


In [34]:
# Inject the needs that effects add as *conditional* (gated) needs. A need
# referenced by a buff's additionalNeedsDemand is consumed by the target
# residence ONLY while the granting effect is active, so it must appear on the
# residence exactly once, gated behind that effect (requiresItem = effect guid),
# and never as a plain ungated base need.
#
# Generalised over ALL effects (mythical villa items *and* monument / colosseum /
# hippodrome island-event effects), derived from each effect's buffs rather than a
# hardcoded item->need mapping. The conditional need's consumption rate lives on
# the residence's own NeedsList entry (the asset marks such needs
# IsOnlyAvailableThroughBuff with a real rate), so we convert that base entry in
# place: the real rate is preserved and no duplicate ungated copy remains.
buildingBuffs_by_guid = {b["guid"]: b for b in params["buildingBuffs"]}
residence_by_guid = {r["guid"]: r for r in params["residenceBuildings"]}

for js in effect_json.values():
    effect_guid = js["guid"]
    need_guids = list(dict.fromkeys(
        need_guid
        for buff_guid in js["buffs"]
        for need_guid in buildingBuffs_by_guid.get(buff_guid, {}).get("additionalNeedsDemand", [])
    ))
    if not need_guids:
        continue

    for need_guid in need_guids:
        for target_guid in js["targets"]:
            residence = residence_by_guid.get(target_guid)
            if residence is None:
                # additionalNeedsDemand only applies to residences; an effect may
                # also target non-residence buildings via its other buffs -> skip.
                continue

            existing = next((n for n in residence["needsList"] if n["need"] == need_guid), None)
            if existing is not None:
                prev = existing.get("requiresItem")
                if prev is not None and prev != effect_guid:
                    print(residence["guid"], need_guid, "already gated by", prev,
                          "- also targeted by effect", effect_guid, "(cross-wired tiers?)")
                # Convert the base need into a gated one in place, preserving its
                # real consumption rate; do NOT leave a second ungated copy behind.
                existing["requiresItem"] = effect_guid
            else:
                # Need not present on the residence at all: gate a fresh entry.
                # (Rate normally lives on the residence, so this is only a fallback.)
                residence["needsList"].append({
                    "need": need_guid,
                    "needConsumptionRate": None,
                    "requiresItem": effect_guid,
                })

add_parameter("residenceBuildings", params["residenceBuildings"], "ResidenceBuilding", "Residence building for a population level")

### Techs (must be after buffs)

In [35]:
result = []
for asset in templates["Tech"].assets:   
    
    cfg = asset.Tech    

    effects = [effect.EffectAsset.guid for effect in cfg.Rewards.Effects if effect.EffectAsset.guid  in relevant_effects]
       
    if len(effects):
        js = icons.process(asset)
        js["locaText"] = cfg.TechName().values
        js["effects"] = effects
        js["isRepeatable"] = cfg.IsRepeatable()
        js["dlcUnlocks"] = get_unlocks(asset)

        for effect_guid in effects:
            effect_json[effect_guid]["source"] = "tech"

        result.append(js)
add_parameter("techs", result, "Tech", "Technologies from the discovery tree with relevant effects")

### Religion

In [36]:
result = []
for asset in templates["Patron"].assets:   
    
    cfg = asset.Patron 
    js = icons.process(asset)
    js["locaText"] = cfg.PatronName().values 
    js["dlcUnlocks"] =  get_unlocks(asset)
    js["wonder"] = cfg.Wonder.guid if cfg.Wonder.guid in relevant_effects else None
    if js["wonder"] in effect_json:
        effect_json[js["wonder"]]["effectScope"] = "ObjectsInMeta"
        effect_json[js["wonder"]]["source"] = "veneration-effect"
    js["dominantEffects"] = [item.GUID.guid for item in cfg.DominantEffects if item.GUID.guid in relevant_effects]
    js["localEffects"] = []
    for entry in cfg.LocalEffects:
        if entry.GUID.guid not in relevant_effects:
            continue

        js["localEffects"].append({
            "effect": entry.GUID.guid,
            "milestones": [{
                "devotion": item.Devotion(),
                "buffScaling": item.BuffScaling()
            } for item in entry.Milestones],
            "title": entry.Title().values
        })
        

    result.append(js)
add_parameter("patrons", result, "Patrons", "Patrons with effects increased by devotion.")

### Festivals

In [37]:
for asset in templates["Festival"].assets:  
    try:
        cfg = asset.Festival
        effects = [effect.Effect.guid for effect in cfg.FestivalEffects if effect.Effect.guid  in relevant_effects]
        if len(effects):
            print(asset.Standard.Name)
            
            duration = cfg.FestivalDuration().seconds
            for effect in effects:
                js = effect_json[effect]
                js["effectDuration"] = duration
                js["source"] = "festival"

    except Exception as e:
        print(asset.guid, e)

Name: Festival Cernunnos
Name: Festival Minerva
Name: Festival Vulcan


### Items

In [38]:
result = []
for template in ["Item", "ItemWithBoost"]:
    for asset in templates[template].assets:
        try:
            cfg = asset.Effect
            buffs = [buff.GUID.guid for buff in cfg.Buffs if buff.GUID.guid  in relevant_buffs]
            if len(buffs):

                js = icons.process(asset)
                js["buffs"] = buffs
                boost_attr = asset.find("ItemWithBoost.BoostBuffs")
                if isinstance(boost_attr, ListAttribute):
                    boost_buffs = []
                    for boost_entry in boost_attr:
                        boost_ref = boost_entry.find_ref("GUID")
                        if boost_ref is not None and boost_ref.guid in relevant_buffs:
                            boost_buffs.append(boost_ref.guid)
                    if boost_buffs:
                        js["boostBuffs"] = boost_buffs
                js["targets"] = [guid for pool in cfg.Targets for guid in flatten_pool(pool.GUID())]
                
                if len(js["targets"]) == 0:
                    print(asset, asset.guid, "has no targets")

                js["effectScope"] = cfg.EffectScope()
                js["excludeEffectSourceGUID"] = cfg.ExcludeEffectSourceGUID()
                js["rarity"] = asset.Item.Rarity()
                js["dlcUnlocks"] = get_unlocks(asset)

                filter_dlc_targets(js)
                filter_effect_targets(js)

                if len(cfg.ExcludeFromTargets._value_list) > 0:
                    print(asset, asset.guid, "ExcludeFromTargets", cfg.ExcludeFromTargets())

                #if not asset.Item.OnlyEquippableOnce():
                #    print(asset, asset.guid, "OnlyEquippableOnce", asset.Item.OnlyEquippableOnce())

                relevant_effects.add(asset.guid)
                result.append(js)
        except Exception as e:
            print(asset.guid, e)

add_parameter("items", result, "Item", "Items equipable in Villa with relevant buffs.")

### Icons (after all other parameters are processed)

In [39]:
add_parameter("icons", icons.to_dict(), "Icon", "Icon data URLs indexed by icon identifier")

In [40]:
# Override the icons schema to use additionalProperties instead of individual properties
schema["properties"]["icons"] = {
    "type": "object",
    "title": "Icon",
    "description": "Icon data URLs indexed by icon identifier",
    "additionalProperties": {
        "type": "string"
    }
}

## Texts

In [41]:
result = []

for referenceName, lineID in {
    "activeIslandEffects": -6901888795121698763,
    "affectedBuildings": -6904082780390519730,
    "all": -6915762395677959303,
    "allBuildings": -6905197213145912196,
    "allIslands": -6901911865748271091,
    "apply": -6911313214704811434,
    "areaEffects": -6903712981431318990,
    "belief": -6908820605821366843,
    "buffs": -6907330109749815048,
    "buildings": -6917203094771649915,
    "confirm": -6915627349826723809,
    "constructionMaterial": -6917180467021169596,
    "consumption": -6902845924876156586,
    "currentPatron": -6913489339087789708,
    "devotion": -6913943659840406396, # devotion/belief
    "discovery": -6913479974945708503,
    "download": -6912820157918341621,
    "effect": -6902234897441092053,
    "effects": -6902362035358284768,
    "eventDuration": -6902018417385309297,
    "festival": -6908773579491322283,
    "fromAreaEffectsAndSpecialists": -6902480799995931355,
    "fuelEfficiency": -6901428646395682482,
    "global": -6910306885876902499,
    "globalEffects": -6900227375200473257,
    "goods": -6904656400857447148,
    "goodsConsumption": -6916926126237868583,
    "help": -6906640699676227597,
    "heroicSpecialist": -6903778921973072056,
    "islandBuffs": -6901814024921012623,
    "islandWideEffects": -6911394270208630347,
    "language": -6910251369175148580,
    "needAttributes": -6913212391033157055,
    "needConsumption": -6901164581421416158,
    "needs": -6915455774919739315,
    "noPatron": -6899884938127726030,
    "outputStorage": -6913551300295748472,
    "patron": -6904053276046802194,
    "patronEffects": -6916892856177889166,
    "production": -6914202634429573508,
    "productionBuildings": -6914034826827989276,
    "productionChain": -6910138230344138817,
    "productivity": -6902990997164434871,
    "publicBuildings": -6899952445988598006,
    "residences": -6908559383606239043, # or -6907998875214561561, -6901535596004234866
    "residents": -6900991602074423381,
    "runningEvent": -6900991734456921008,
    "showInformation": -6915422247772755926,
    "showNeedsOfPopulationTier": -6911349978207385087,
    "silo": -6907773021498254428,
    "settings": -6909909211298253262,
    "talentumPerMinute": -6912455514594073319,
    "traders": -6903619873875452438,
    "trading": -6911891323366335933,
    "total": -6912046064850205942,
    "tradeRoutes": -6915569607474692589,
    "venerationEffects": -6915910452867912431,
    "wonderEffect": -6905498542987856790,
    "world": -6907309872814266407,
    "workforce": -6914935202834947869
}.items():
    try:
        result.append({
            "name": referenceName,
            "lineID": lineID,
            "locaText": assets.texts.get(lineID).values
        })

        if referenceName == "global":
            for session in params["sessions"]:
                if session["guid"] == 37135:
                    session["locaText"] = result[-1]["locaText"]
                    break
    except:
        print(f"No localization found for {referenceName} with Line ID {lineID}")


add_parameter("texts", result, "Text", "Texts from the game with localization.")

## Ensure params is serializable

In [42]:
from typing import Dict
from assetextractor.parsing.core.attributes import Attribute


def find_unprocessed(data: t.Any, path: str = "root"):
    if isinstance(data, Attribute) or isinstance(data, Asset):
        raise ValueError(f"{path} is of type {data.__class__}")

    if isinstance(data, dict):
        for key in data:    
            if not isinstance(key, str):
                raise ValueError(f"{path}.{key} is not a string")     
            find_unprocessed(data[key], f"{path}.{key}")
    elif isinstance(data, list):
        for i, item in enumerate(data):
            find_unprocessed(item, f"{path}[{i}]")
    else:
        # Base case: primitive value
        pass

find_unprocessed(params)

### Save parameters and schema

In [43]:
with open("../anno-117-calculator/js/params.js", "w", encoding="utf-8") as f:
    f.write('if(window.params == null)window.params=')
    f.write(json.dumps(params, ensure_ascii=False, indent=2, sort_keys=True))

In [44]:

# Save schema to the same directory as params.js
with open("../anno-117-calculator/js/params.schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

print("Finished")

Finished
